# Lab 01 - Configuración Inicial y Workspace

**Objetivos:**
- Familiarizarse con notebooks de Databricks
- Crear DataFrames básicos
- Ejecutar transformaciones simples
- Trabajar con diferentes lenguajes

## Parte 1: Crear DataFrame Simple

In [ ]:
from pyspark.sql.functions import col, upper, when, current_timestamp

# Crear DataFrame de ejemplo
data = [
    (1, "Alice", "Ventas", 75000),
    (2, "Bob", "IT", 85000),
    (3, "Charlie", "Ventas", 65000),
    (4, "Diana", "Marketing", 70000),
    (5, "Eve", "IT", 90000)
]

columns = ["id", "nombre", "departamento", "salario"]

df = spark.createDataFrame(data, columns)

print(f"✅ DataFrame creado con {df.count()} registros")
display(df)

## Parte 2: Transformaciones Básicas

In [ ]:
# Transformaciones
df_transformed = df \
    .withColumn("nombre_upper", upper(col("nombre"))) \
    .withColumn("nivel_salarial",
                when(col("salario") < 70000, "Junior")
                .when(col("salario") < 80000, "Mid")
                .otherwise("Senior")) \
    .withColumn("fecha_proceso", current_timestamp())

display(df_transformed)

## Parte 3: Filtros y Agregaciones

In [ ]:
# Filtrar empleados de IT
df_it = df_transformed.filter(col("departamento") == "IT")
print(f"Empleados de IT: {df_it.count()}")
display(df_it)

In [ ]:
# Agregaciones por departamento
from pyspark.sql.functions import avg, count, sum as _sum

df_summary = df.groupBy("departamento").agg(
    count("*").alias("num_empleados"),
    avg("salario").alias("salario_promedio"),
    _sum("salario").alias("total_salarios")
).orderBy("salario_promedio", ascending=False)

display(df_summary)

## Parte 4: Guardar Datos

In [ ]:
# Guardar en formato Delta
output_path = "/tmp/lab01/empleados"

df_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

print(f"✅ Datos guardados en: {output_path}")

In [ ]:
# Leer datos guardados
df_read = spark.read.format("delta").load(output_path)
print(f"📊 Registros leídos: {df_read.count()}")
display(df_read)

## Parte 5: SQL Queries

In [ ]:
# Crear temp view
df_transformed.createOrReplaceTempView("empleados")
print("✅ Temp view 'empleados' creada")

In [ ]:
%%sql
SELECT 
    departamento,
    nivel_salarial,
    COUNT(*) as num_empleados,
    AVG(salario) as salario_promedio
FROM empleados
GROUP BY departamento, nivel_salarial
ORDER BY departamento, salario_promedio DESC

## ✅ Lab Completado

Has aprendido:
- ✅ Crear DataFrames
- ✅ Aplicar transformaciones
- ✅ Filtros y agregaciones
- ✅ Guardar en formato Delta
- ✅ Queries SQL